# Extended Practice: Filtering in SQL

Use these exercises to deepen your skills with **row filtering** (`WHERE`).  
This lab builds on the original optional practice and adds more patterns you will use daily as a data analyst / data engineer.

## Learning objectives

By the end of this notebook you should be able to:

- Filter with comparison operators (`>`, `<`, `=`, `!=`, `>=`, `<=`)
- Combine conditions with `AND`, `OR`, and parentheses
- Use `IN`, `BETWEEN`, `LIKE`, `IS NULL` / `IS NOT NULL`
- Apply pattern matching with wildcards `%` and `_`
- Combine filters with `ORDER BY` and `LIMIT`
- Read and write clean, readable filter logic

## Database

You will work with a **digital media store** database (Chinook-style schema).  
Table and column names below match the ones used in the original practice exercises (`tracks`, `employees`, `invoices`, etc.).  
If your environment uses different casing (`Track` vs `tracks`), adjust the names accordingly.

### Relevant tables (simplified)

| Table | Key columns |
|-------|-------------|
| **tracks** | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer`, `Milliseconds`, `Bytes`, `UnitPrice` |
| **employees** | `EmployeeId`, `LastName`, `FirstName`, `Title`, `ReportsTo`, `BirthDate`, `HireDate`, `City`, `State`, `Country`, `Email` |
| **invoices** | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingAddress`, `BillingCity`, `BillingState`, `BillingCountry`, `BillingPostalCode`, `Total` |
| **customers** | `CustomerId`, `FirstName`, `LastName`, `Company`, `City`, `State`, `Country`, `Email`, `SupportRepId` |
| **albums** | `AlbumId`, `Title`, `ArtistId` |
| **artists** | `ArtistId`, `Name` |
| **genres** | `GenreId`, `Name` |
| **media_types** | `MediaTypeId`, `Name` |

> Tip: Always start by exploring with `SELECT * FROM table LIMIT 5;` when you are unsure of column names or value ranges.

---
# Part 1 — Core filtering patterns

These exercises cover the most common filters: comparisons, `IN`, `BETWEEN`, compound conditions, and `LIKE`.

## Exercise 1: Track duration

The content team is analyzing track length to build a playlist of longer songs.  
Find all tracks longer than **5 minutes** (300,000 milliseconds).

### Instructions

Write a SQLite query that:

- Selects `Name` and `Milliseconds` from `tracks`
- Keeps only rows where `Milliseconds > 300000`

**Expected output (first rows):**

```
+-----------------------------------------+--------------+
| Name                                    | Milliseconds |
+-----------------------------------------+--------------+
| For Those About To Rock (We Salute You) |       343719 |
| Balls to the Wall                       |       342562 |
| Princess of the Dawn                    |       375418 |
| Go Down                                 |       331180 |
| Let There Be Rock                       |       366654 |
...
(Output limit exceeded; many rows match)
```

**Hints**

- `SELECT col1, col2 FROM table WHERE condition;`
- Comparison operators: `>`, `<`, `=`, `!=` (or `<>`), `>=`, `<=`

## Exercise 2: Employees by city (`IN`)

HR is planning an office event and needs employees based in specific cities.  
Find employees living in **Edmonton** or **Calgary**.

### Instructions

- Select `FirstName`, `LastName`, and `City` from `employees`
- Filter with `City IN ('Edmonton', 'Calgary')`

**Expected output:**

```
+-----------+----------+----------+
| FirstName | LastName | City     |
+-----------+----------+----------+
| Andrew    | Adams    | Edmonton |
| Nancy     | Edwards  | Calgary  |
| Jane      | Peacock  | Calgary  |
| Margaret  | Park     | Calgary  |
| Steve     | Johnson  | Calgary  |
| Michael   | Mitchell | Calgary  |
+-----------+----------+----------+
```

**Hints**

- `WHERE col IN (value1, value2, ...)` is cleaner than multiple `OR`s
- String literals use single quotes in standard SQL (SQLite also accepts double quotes)

## Exercise 3: Quarterly invoices (`BETWEEN`)

Finance needs invoices from the first quarter of 2010.  
Retrieve invoices with `InvoiceDate` between **2010-01-01** and **2010-03-31** (inclusive).

### Instructions

- Select `InvoiceId`, `InvoiceDate`, and `Total` from `invoices`
- Use `BETWEEN ... AND ...` on `InvoiceDate`

**Expected output (sample):**

```
+-----------+---------------------+-------+
| InvoiceId | InvoiceDate         | Total |
+-----------+---------------------+-------+
|        84 | 2010-01-08 00:00:00 |  1.98 |
|        85 | 2010-01-08 00:00:00 |  1.98 |
|        86 | 2010-01-09 00:00:00 |  3.96 |
...
|       104 | 2010-03-29 00:00:00 |  0.99 |
+-----------+---------------------+-------+
```

**Hints**

- `WHERE col BETWEEN low AND high` is inclusive on both ends
- Dates are usually stored as text or datetime; string comparison works for ISO format (`YYYY-MM-DD`)

## Exercise 4: Compound conditions (`AND`)

Marketing wants tracks that are **longer than 4 minutes** *and* **priced under $1.99**.

### Instructions

- Select `Name`, `Milliseconds`, and `UnitPrice` from `tracks`
- Filter: `Milliseconds > 240000` **AND** `UnitPrice < 1.99`

**Expected output (first rows):**

```
+-----------------------------------------+--------------+-----------+
| Name                                    | Milliseconds | UnitPrice |
+-----------------------------------------+--------------+-----------+
| For Those About To Rock (We Salute You) |       343719 |      0.99 |
| Balls to the Wall                       |       342562 |      0.99 |
| Restless and Wild                       |       252051 |      0.99 |
...
```

**Hints**

- Both conditions must be true → use `AND`
- Parentheses are optional here but become important with mixed `AND`/`OR`

## Exercise 5: Grouping conditions (`AND` + `OR` / `IN`)

Sales wants invoices from the **USA or Canada** whose **Total is greater than $50**.

### Instructions

- Select `InvoiceId`, `BillingCountry`, and `Total` from `invoices`
- Keep rows where billing country is USA or Canada **and** total > 50

**Expected output:**

```
+-----------+----------------+-------+
| InvoiceId | BillingCountry | Total |
+-----------+----------------+-------+
(Zero rows)
```

In the classic Chinook dataset there are no invoices from USA/Canada with Total > 50.  
That is a valid and useful result — always check edge cases.

**Hints**

- Prefer `BillingCountry IN ('USA', 'Canada') AND Total > 50`
- If you write it with `OR`, wrap the country condition in parentheses:
  `(BillingCountry = 'USA' OR BillingCountry = 'Canada') AND Total > 50`

## Exercise 6: Pattern matching (`LIKE`)

HR wants every employee whose title **starts with** `"Sales"`.

### Instructions

- Select `LastName`, `FirstName`, and `Title` from `employees`
- Filter with `Title LIKE 'Sales%'`

**Expected output:**

```
+----------+-----------+---------------------+
| LastName | FirstName | Title               |
+----------+-----------+---------------------+
| Edwards  | Nancy     | Sales Manager       |
| Peacock  | Jane      | Sales Support Agent |
| Park     | Margaret  | Sales Support Agent |
| Johnson  | Steve     | Sales Support Agent |
+----------+-----------+---------------------+
```

**Hints**

- `%` = any sequence of characters (including empty)
- `_` = exactly one character
- `LIKE` is case-insensitive in SQLite by default for ASCII

---
# Part 2 — Extended filtering skills

Practice additional operators and combinations that appear constantly in real queries.

## Exercise 7: Exclusion with `NOT IN` / `!=`

Find all employees who do **not** live in Calgary.

### Instructions

- Select `FirstName`, `LastName`, `City` from `employees`
- Exclude rows where `City = 'Calgary'`

**Hints**

- `WHERE City != 'Calgary'` or `WHERE City <> 'Calgary'`
- Or `WHERE City NOT IN ('Calgary')`
- Be careful with `NULL`s: `!=` does not match `NULL` rows

## Exercise 8: Missing values (`IS NULL` / `IS NOT NULL`)

Some tracks have no composer listed.  
Find tracks where `Composer` **is NULL**, and return only `Name` and `Composer`.  
Limit the result to the first 15 rows for readability.

### Instructions

- `SELECT Name, Composer FROM tracks WHERE Composer IS NULL LIMIT 15;`

**Hints**

- Never write `WHERE Composer = NULL` — it always returns empty
- Correct forms: `IS NULL` / `IS NOT NULL`

## Exercise 9: Numeric range with `BETWEEN` + extra filter

Find tracks whose duration is between **3 and 4 minutes** (180000–240000 ms) **and** whose unit price is exactly **0.99**.

### Instructions

- Select `Name`, `Milliseconds`, `UnitPrice`
- `Milliseconds BETWEEN 180000 AND 240000 AND UnitPrice = 0.99`

**Hints**

- `BETWEEN` is inclusive
- You can mix `BETWEEN` with other predicates using `AND`/`OR`

## Exercise 10: Flexible string patterns

Find customers whose **last name** contains the letter sequence `"son"` (anywhere in the name).

### Instructions

- Select `FirstName`, `LastName`, `Country` from `customers`
- `LastName LIKE '%son%'`

**Hints**

- `%son%` matches "Johnson", "Jackson", "Wilson", etc.
- Try also: names that **start** with "A" → `'A%'`; names that are exactly 5 characters → `'_____'`

## Exercise 11: Date filtering without `BETWEEN`

Retrieve invoices issued **strictly after** 2012-12-31 (i.e. in 2013 or later).  
Return `InvoiceId`, `InvoiceDate`, `Total`, ordered by date ascending.

### Instructions

- Filter with `InvoiceDate > '2012-12-31'`
- Add `ORDER BY InvoiceDate`

**Hints**

- ISO date strings sort correctly with ordinary comparison operators
- You can also write `InvoiceDate >= '2013-01-01'`

## Exercise 12: Multiple cities + high totals

Find invoices billed to **Brazil, Germany, or France** with a total of **at least 10**.

### Instructions

- Select `InvoiceId`, `BillingCountry`, `Total`
- `BillingCountry IN ('Brazil', 'Germany', 'France') AND Total >= 10`

**Hints**

- Keep the `IN` list readable; one value per line is fine in many SQL styles

## Exercise 13: Combining `OR` carefully

Return tracks that are either:

- longer than 10 minutes (`Milliseconds > 600000`), **or**
- more expensive than $1.50 (`UnitPrice > 1.50`)

Select `Name`, `Milliseconds`, `UnitPrice` and order by `Milliseconds` descending.  
Limit to 20 rows.

### Instructions

Write the `OR` condition and add `ORDER BY` + `LIMIT`.

**Hints**

- When mixing `AND` and `OR` later, always use parentheses to control precedence  
  (`AND` binds tighter than `OR` in SQL)

## Exercise 14: Employees hired in a year range

Find employees hired between **2002-01-01** and **2003-12-31**.  
Return `FirstName`, `LastName`, `Title`, `HireDate`.

### Instructions

- Use `HireDate BETWEEN '2002-01-01' AND '2003-12-31'`

**Hints**

- Same `BETWEEN` pattern you used for invoices works for hire dates

## Exercise 15: Negative pattern (`NOT LIKE`)

Find all employee titles that do **not** start with `"Sales"`.

### Instructions

- Select `LastName`, `FirstName`, `Title`
- `Title NOT LIKE 'Sales%'`

**Hints**

- `NOT LIKE` is the natural counterpart of `LIKE`

---
# Part 3 — Challenge problems

These require combining several ideas. Try them before looking at the solutions.

## Challenge A: Promotion candidates

Marketing wants a shortlist of tracks that satisfy **all** of the following:

1. Duration between 4 and 6 minutes (240000–360000 ms)
2. Unit price exactly 0.99
3. Name contains the word `"Love"` (case-insensitive is fine)

Return `Name`, `Milliseconds`, `UnitPrice`.  
Order by duration descending.

## Challenge B: High-value international invoices

Find invoices where:

- Billing country is **not** `'USA'`
- Total is greater than **15**

Return `InvoiceId`, `BillingCountry`, `BillingCity`, `Total`.  
Order by `Total` descending and show only the top 25.

## Challenge C: Support team in Canada

List employees who:

- Have `"Support"` somewhere in their title, **and**
- Live in a Canadian city (the cities present in the data are Edmonton, Calgary, Lethbridge)

Return `FirstName`, `LastName`, `Title`, `City`.

## Challenge D: Customers with incomplete company info

Find customers whose `Company` is NULL **and** who live in either `'USA'` or `'Canada'`.  
Return `FirstName`, `LastName`, `Country`, `City`, `Company`.  
Order by country, then last name.

## Challenge E: Very long or very short tracks

Return tracks that are either:

- shorter than 1 minute (`Milliseconds < 60000`), **or**
- longer than 15 minutes (`Milliseconds > 900000`)

Select `Name`, `Milliseconds`, `UnitPrice`.  
Order by `Milliseconds` ascending.

---
# Quick reference — Filtering cheatsheet

```sql
-- Comparisons
WHERE col >  10
WHERE col >= 10
WHERE col =  'text'
WHERE col != 'text'          -- or <> 

-- Ranges
WHERE col BETWEEN 10 AND 20  -- inclusive
WHERE col NOT BETWEEN 10 AND 20

-- Sets
WHERE col IN ('A', 'B', 'C')
WHERE col NOT IN ('A', 'B')

-- Nulls (special syntax!)
WHERE col IS NULL
WHERE col IS NOT NULL

-- Pattern matching
WHERE col LIKE 'prefix%'     -- starts with
WHERE col LIKE '%suffix'     -- ends with
WHERE col LIKE '%middle%'    -- contains
WHERE col LIKE '_a%'         -- second char is 'a'
WHERE col NOT LIKE 'x%'

-- Combining
WHERE cond1 AND cond2
WHERE cond1 OR  cond2
WHERE (cond1 OR cond2) AND cond3   -- parentheses control order

-- Often used together
SELECT ...
FROM table
WHERE ...
ORDER BY col DESC
LIMIT 20;
```

### Common pitfalls

1. `WHERE col = NULL` → always empty. Use `IS NULL`.
2. Forgetting parentheses with mixed `AND`/`OR`.
3. Using double quotes for string values in strict SQL modes (prefer single quotes).
4. Assuming `LIKE` is case-sensitive (SQLite is usually case-insensitive for ASCII).

---
# Solutions

Try each exercise first. Use this section only to check or unblock yourself.

## Part 1 solutions

### Exercise 1
```sql
SELECT Name, Milliseconds
FROM tracks
WHERE Milliseconds > 300000;
```

### Exercise 2
```sql
SELECT FirstName, LastName, City
FROM employees
WHERE City IN ('Edmonton', 'Calgary');
```

### Exercise 3
```sql
SELECT InvoiceId, InvoiceDate, Total
FROM invoices
WHERE InvoiceDate BETWEEN '2010-01-01' AND '2010-03-31';
```

### Exercise 4
```sql
SELECT Name, Milliseconds, UnitPrice
FROM tracks
WHERE Milliseconds > 240000
  AND UnitPrice < 1.99;
```

### Exercise 5
```sql
SELECT InvoiceId, BillingCountry, Total
FROM invoices
WHERE BillingCountry IN ('USA', 'Canada')
  AND Total > 50;
```

### Exercise 6
```sql
SELECT LastName, FirstName, Title
FROM employees
WHERE Title LIKE 'Sales%';
```

## Part 2 solutions

### Exercise 7
```sql
SELECT FirstName, LastName, City
FROM employees
WHERE City != 'Calgary';
-- alternative: WHERE City NOT IN ('Calgary');
```

### Exercise 8
```sql
SELECT Name, Composer
FROM tracks
WHERE Composer IS NULL
LIMIT 15;
```

### Exercise 9
```sql
SELECT Name, Milliseconds, UnitPrice
FROM tracks
WHERE Milliseconds BETWEEN 180000 AND 240000
  AND UnitPrice = 0.99;
```

### Exercise 10
```sql
SELECT FirstName, LastName, Country
FROM customers
WHERE LastName LIKE '%son%';
```

### Exercise 11
```sql
SELECT InvoiceId, InvoiceDate, Total
FROM invoices
WHERE InvoiceDate > '2012-12-31'
ORDER BY InvoiceDate;
```

### Exercise 12
```sql
SELECT InvoiceId, BillingCountry, Total
FROM invoices
WHERE BillingCountry IN ('Brazil', 'Germany', 'France')
  AND Total >= 10;
```

### Exercise 13
```sql
SELECT Name, Milliseconds, UnitPrice
FROM tracks
WHERE Milliseconds > 600000
   OR UnitPrice > 1.50
ORDER BY Milliseconds DESC
LIMIT 20;
```

### Exercise 14
```sql
SELECT FirstName, LastName, Title, HireDate
FROM employees
WHERE HireDate BETWEEN '2002-01-01' AND '2003-12-31';
```

### Exercise 15
```sql
SELECT LastName, FirstName, Title
FROM employees
WHERE Title NOT LIKE 'Sales%';
```

## Challenge solutions

### Challenge A
```sql
SELECT Name, Milliseconds, UnitPrice
FROM tracks
WHERE Milliseconds BETWEEN 240000 AND 360000
  AND UnitPrice = 0.99
  AND Name LIKE '%Love%'
ORDER BY Milliseconds DESC;
```

### Challenge B
```sql
SELECT InvoiceId, BillingCountry, BillingCity, Total
FROM invoices
WHERE BillingCountry != 'USA'
  AND Total > 15
ORDER BY Total DESC
LIMIT 25;
```

### Challenge C
```sql
SELECT FirstName, LastName, Title, City
FROM employees
WHERE Title LIKE '%Support%'
  AND City IN ('Edmonton', 'Calgary', 'Lethbridge');
```

### Challenge D
```sql
SELECT FirstName, LastName, Country, City, Company
FROM customers
WHERE Company IS NULL
  AND Country IN ('USA', 'Canada')
ORDER BY Country, LastName;
```

### Challenge E
```sql
SELECT Name, Milliseconds, UnitPrice
FROM tracks
WHERE Milliseconds < 60000
   OR Milliseconds > 900000
ORDER BY Milliseconds ASC;
```

---
# Next steps

- Re-run the same filters with `COUNT(*)` to see how selective each condition is.
- Add `ORDER BY` and `LIMIT` to every query until it becomes automatic.
- Practice rewriting `IN` lists as multiple `OR`s (and the reverse) so you understand equivalence.
- When you learn joins, come back and filter across tables (e.g. tracks of a given genre, invoices of customers from a country).

Happy querying!